# Estimating individual baselines

Fix the interaction effects. Vary baselines, and calculate probability of observing the transition from 1 --> 2, for each individual. Pick the baselines which maximise this likelihood.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from ising.model import UpdateMethod
from scipy.special import expit
from scipy.stats import norm
from scipy.stats.qmc import LatinHypercube

from climate_attitudes.dataset import Dataset
from climate_attitudes.settings import Config
from climate_attitudes.visualisation import configure_mpl
from ising import Ising

configure_mpl(Path("../fonts/"))

np.set_printoptions(linewidth=200)

RANDOM_SEED = 202606081503

In [ ]:
config = Config(_env_file="../.env")
dataset = Dataset.load(
    config,
    name="reduced_no_imputation",
    with_imputation=False,
    verbose=False,
)
_, Y, X = dataset.indices_to_numpy(kind="time-series", binarise=True, seed=RANDOM_SEED)

In [ ]:
rng = np.random.default_rng(RANDOM_SEED)
model = Ising.fit(
    Y,
    update_method=UpdateMethod.SYNCHRONOUS,
    rng=rng,
    node_labels=dataset.schema.get_short_names("measurement"),
    self_loops=True,
)

In [ ]:
sample_individuals = rng.choice(np.arange(Y.shape[0]), size=10, replace=False)
sample_Y = Y[sample_individuals]

In [ ]:
nsamples = 2809
lhs = LatinHypercube(d=model.size, strength=2, rng=rng.spawn(1)[0])
samples = lhs.random(nsamples)
candidates = norm.ppf(samples, loc=model.h, scale=1)

In [ ]:
log_prob = np.zeros(nsamples, dtype=np.float64)
for i, maybe_h in enumerate(candidates):
    log_prob[i] = -(
        model.time_series_nll_sync(
            sample_Y[:1], np.ones((1, 2)), maybe_h, model.j, model.adj
        )
        + model.time_series_nll_sync(
            sample_Y[:1, ::-1], np.ones((1, 2)), maybe_h, model.j, model.adj
        )
    )

orig_idx = np.argmax(log_prob)
orig_h = candidates[orig_idx]
candidates[np.argmax(log_prob)]

In [ ]:
np.exp(log_prob[orig_idx])

In [ ]:
dataset.schema.get_short_names("measurement")

In [ ]:
expit(candidates[np.argmax(log_prob)])

In [ ]:
sample_Y[0]

In [ ]:
alt = sample_Y[1:2].copy()
# alt[0,0,5] = 1

log_prob = np.zeros(nsamples, dtype=np.float64)
for i, maybe_h in enumerate(candidates):
    log_prob[i] = -(
        model.time_series_nll_sync(alt, np.ones((1, 2)), maybe_h, model.j, model.adj)
        + model.time_series_nll_sync(
            alt[:, ::-1], np.ones((1, 2)), maybe_h, model.j, model.adj
        )
    )

alt_h = candidates[np.argmax(log_prob)]
candidates[np.argmax(log_prob)]

In [ ]:
expit(candidates[np.argmax(log_prob)])

In [ ]:
np.exp(log_prob[orig_idx])

In [ ]:
fig, axes = plt.subplots(figsize=(10, 4), ncols=2, constrained_layout=True)

vlim = max(abs(orig_h).max(), abs(alt_h).max())

orig_model = model.clone(rng=rng.spawn(1)[0])
orig_model.adj[abs(orig_model.j) < 0.125] = 0
orig_model.h = orig_h
orig_model.draw(seed=20260615, ax=axes[0], vlim_h=(-vlim, vlim))

alt_model = orig_model.clone(rng.spawn(1)[0])
alt_model.h = alt_h
alt_model.draw(
    seed=20260615, ax=axes[1], use_layout_from=orig_model, vlim_h=(-vlim, vlim)
)

In [ ]:
orig_model.draw_state(sample_Y[0, -1], seed=20260615, use_layout_from=orig_model)